# N157 · Pandas选择、缺失与类型

**目标：** 在保留数据类型和索引语义下清洗表。

**先修：** N005, N007, N010。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** DataFrame/Series；loc/iloc；布尔筛选；NA；astype；去重；链式赋值风险。

## 从问题到算法

表格操作首先是**数据契约**：列名是什么，缺失表示什么，重复保留哪一行，结果索引是否有意义？不能只检查屏幕上看起来相同。

从学生记录列表创建表；删除重复邮件时选定“保留首次出现”；删除缺失姓名时仅检查 `name`，不顺便删掉其他列有缺失的学生。`None`、`NaN` 由 `isna`/`dropna` 处理，空字符串则是另一种业务状态。

这里把成绩转为 Pandas 的可空整数 `Int64`，与 NumPy 非可空 `int64` 区别大小写。它能保留缺失值，却拒绝把非整数分数无声截断。每个函数返回新表，不修改输入。

## 最小实现

**本章接口：** `create_dataframe(student_data)`、`drop_duplicate_emails(customers)`、`drop_missing_names(students)`、`change_grade_type(students)`

In [1]:
import pandas as pd
from pandas.testing import assert_frame_equal

def create_dataframe(student_data):
    """Rows are (student_id, age); an empty input keeps both integer columns."""
    return pd.DataFrame(student_data, columns=['student_id', 'age']).astype(
        {'student_id': 'int64', 'age': 'int64'})

def drop_duplicate_emails(customers):
    # First occurrence wins, including the first missing email.
    return customers.drop_duplicates(subset=['email'], keep='first').reset_index(drop=True)

def drop_missing_names(students):
    # An empty string is present, not a missing value.
    return students.dropna(subset=['name']).reset_index(drop=True)

def change_grade_type(students):
    """Teaching contract: nullable integer Int64 preserves missing grades."""
    result = students.copy()
    result['grade'] = result['grade'].astype('Int64')
    return result

## 正确性、前提与复杂度

筛选只改变行集合；去重的输出是每个邮件等价类第一次出现的位置。显式 `reset_index` 声明旧索引不是业务主键。整数转换只改变 `grade`，转换失败是契约错误而不是填 0 的理由。

**代价：** 建表、筛选、类型转换对行数 n 线性；哈希去重通常期望 O(n)。结果需要 O(n) 空间；具体内部实现常数不构成算法复杂度证明。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

raw = pd.DataFrame({'student_id':[1,2,3,4], 'name':['Li',None,'','Zhou'], 'grade':[90.0,81.0,None,75.0]})
clean = change_grade_type(drop_missing_names(raw))
show_table(['stage','rows','index','dtype'], [
    ['before', len(raw), raw.index.tolist(), str(raw.dtypes.to_dict())],
    ['after', len(clean), clean.index.tolist(), str(clean.dtypes.to_dict())]])
display(clean)

stage,rows,index,dtype
before,4,"[0, 1, 2, 3]","{'student_id': dtype('int64'), 'name': dtype('O'), 'grade': dtype('float64')}"
after,3,"[0, 1, 2]","{'student_id': dtype('int64'), 'name': dtype('O'), 'grade': Int64Dtype()}"


,student_id,name,grade
0,1,Li,90
1,3,,<NA>
2,4,Zhou,75


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert_frame_equal(create_dataframe([]), pd.DataFrame({
    'student_id': pd.Series(dtype='int64'), 'age': pd.Series(dtype='int64')}))
assert_frame_equal(create_dataframe([[1,20],[2,21]]), pd.DataFrame({'student_id':[1,2],'age':[20,21]}))
customers = pd.DataFrame({'id':[3,1,2,4], 'email':['a@x','a@x',None,None]})
original = customers.copy(deep=True)
assert_frame_equal(drop_duplicate_emails(customers), customers.iloc[[0,2]].reset_index(drop=True))
assert_frame_equal(customers, original)
students = pd.DataFrame({'name':['a',None,''], 'grade':[90.0,75.0,float('nan')]})
before = students.copy(deep=True)
assert_frame_equal(drop_missing_names(students), students.iloc[[0,2]].reset_index(drop=True))
expected = pd.DataFrame({'name':['a',None,''], 'grade':pd.Series([90,75,None],dtype='Int64')})
assert_frame_equal(change_grade_type(students), expected)
assert_frame_equal(students,before)
try:
    change_grade_type(pd.DataFrame({'grade':[90.5]}))
except (TypeError, ValueError):
    pass
else:
    raise AssertionError('A fractional grade must not be silently truncated')
assert_frame_equal(drop_missing_names(students.iloc[:0]), students.iloc[:0].reset_index(drop=True))
print('N157: 所有本章断言通过')

N157: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 解释NaN不能用==比较。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 区分原地修改和返回新表。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 2877. Create a DataFrame from List（canonical）
- 2882. Drop Duplicate Rows（canonical）
- 2883. Drop Missing Data（canonical）
- 2886. Change Data Type（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。